# Simpson's Paradox: The Group Flips the Story

## Overall metrics can reverse inside groups

Run the cell below first. It enlarges the font for both code and markdown so the notebook is easy to read while walking through it in class.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Rendered markdown */
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.rendered_html {
    font-size: 24px !important;
    line-height: 1.5 !important;
}
.jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 40px !important; }
.jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 34px !important; }
.jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 30px !important; }
.jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 28px !important; }
.jp-RenderedHTMLCommon table, .rendered_html table {
    font-size: 22px !important;
}

/* Code editor (CodeMirror, used by classic + JupyterLab) */
.CodeMirror, .cm-editor, .jp-Editor, .jp-InputArea-editor {
    font-size: 24px !important;
}
.cm-content, .cm-line { font-size: 24px !important; }

/* Code output (print, tracebacks, DataFrame text) */
.jp-OutputArea-output,
.output_area,
.output pre,
.jp-RenderedText pre {
    font-size: 22px !important;
}

/* DataFrame tables in output */
.dataframe, .dataframe th, .dataframe td {
    font-size: 22px !important;
}
</style>
"""))


### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

## The story

Overall, the treatment appears worse. Inside both risk groups, the treatment performs better. **The group variable flips the story.**

## 1. Load the trial data

Rows are patients; columns are risk group, treatment arm, and outcome.

In [ ]:
trial = pd.read_csv("../data/simpsons_paradox_treatment.csv")
trial.head()

In [ ]:
trial.info()

## 2. Inspect the key columns

Identify the comparison, the outcome, and the possible confounder.

In [ ]:
trial[["risk_group", "treatment_arm", "success"]].head()

## 3. Start with counts: `value_counts()`

Counts show whether the groups are balanced.

In [ ]:
trial["treatment_arm"].value_counts()

In [ ]:
trial["risk_group"].value_counts()

## 4. Two-way counts with `pd.crosstab()`

Composition matters *before* outcome comparison.

In [ ]:
pd.crosstab(trial["risk_group"], trial["treatment_arm"])

## 5. The rushed analyst's answer: overall success rate

This is where the rushed analyst starts — and stops too early.

In [ ]:
trial.groupby("treatment_arm")["success"].mean()

## 6. Now stratify by the hidden variable

Group by risk group **and** treatment arm.

In [ ]:
trial.groupby(["risk_group", "treatment_arm"])["success"].mean()

## 7. Use `.agg()` for counts and rates together

A rate without a count is fragile. Show both.

In [ ]:
trial.groupby(["risk_group", "treatment_arm"]).agg(
    patients=("patient_id", "count"),
    success_rate=("success", "mean"),
)

## 8. Make the grouped table readable with `unstack()`

In [ ]:
trial.groupby(["risk_group", "treatment_arm"])["success"].mean().unstack()

## 9. Normalize the crosstab

What fraction of each treatment arm came from each risk group?

In [ ]:
pd.crosstab(trial["risk_group"], trial["treatment_arm"], normalize="columns")

## The four-step Simpson check

1. Overall result
2. Plausible group variable
3. Result inside groups
4. Group composition


## Mini-lab: find the flip

In [ ]:
overall = trial.groupby("treatment_arm")["success"].mean()
stratified = trial.groupby(["risk_group", "treatment_arm"])["success"].mean()
composition = pd.crosstab(trial["risk_group"], trial["treatment_arm"])

print("Overall success rate:")
print(overall)
print("\nStratified success rate:")
print(stratified)
print("\nGroup composition:")
print(composition)

## Discussion

- Which number would be easiest to put in a report?
- Which number would be more honest?
- What is driving the reversal — composition or biology?


## Takeaway

Functions introduced: `groupby`, `.agg`, `pd.crosstab`, `value_counts`.

**Concept learned: important groups can reverse the headline conclusion.**